In [1]:
!uv pip install chromadb

Using Python 3.14.6 environment at: E:\AI-ML\.venv
Checked 1 package in 43ms


In [2]:
!uv pip install pypdf
!uv pip install PyMuPDF

Using Python 3.14.6 environment at: E:\AI-ML\.venv
Checked 1 package in 18ms
Using Python 3.14.6 environment at: E:\AI-ML\.venv
Checked 1 package in 18ms


In [3]:
import numpy as np
import numpy as np
from reviews_data import make_reviews              # the 60 reviews from last class
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

In [4]:
chunks = [
    "the food was good and the service was fast",
    "good food and very friendly staff",
    "great service and great value for money",
    "the momo was delicious and the staff were friendly",
    "delicious food, fast service, good price",
    "excellent food and excellent service",
    "the curry was delicious and the staff was kind",
    "friendly staff and fresh food",
    "fresh ingredients and good flavour",
    "the dal bhat was delicious and hot",
    "good value and the service was quick",
    "quick service and delicious coffee",
    "the staff was friendly and the food was fresh",
    "great flavour and a clean place",
    "clean tables and very good food",
    "the thukpa was excellent and hot",
    "excellent value, the food was fresh",
    "we loved the food, service was fast",
    "the biryani was delicious and the portion was generous",
    "generous portion and good price",
    "the tea was good and the staff smiled",
    "friendly waiter and delicious pastries",
    "the food arrived hot and fresh",
    "hot food, fast service, friendly staff",
    "very good experience, we will come again",
    "the chef was great and the food was delicious",
    "great place, clean and friendly",
    "the salad was fresh and the service was good",
    "good coffee and a quick, friendly staff",
    "delicious flavour and excellent price",
]


In [5]:
# ============================================================
# CHROMA + TF-IDF + SVD SEARCH
# ============================================================

import chromadb

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline


# ============================================================
# 1. CHECK CHUNKS
# ============================================================

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i}:")
    print(chunk)


# ============================================================
# 2. CREATE THE EMBEDDING MODEL
# ============================================================

lsa = make_pipeline(
    TfidfVectorizer(
        stop_words="english"
    ),
    TruncatedSVD(
        n_components=2,
        random_state=0
    )
)


# ============================================================
# 3. EMBED ALL CHUNKS
# ============================================================

chunk_embeddings = lsa.fit_transform(chunks)

print(
    "\nChunk embeddings shape:",
    chunk_embeddings.shape
)


# ============================================================
# 4. FUNCTION TO EMBED A QUERY
# ============================================================

def embed(query):

    embedding = lsa.transform(
        [query]
    )

    print(
        "Query embedding shape:",
        embedding.shape
    )

    return embedding


# ============================================================
# 5. CREATE CHROMA CLIENT
# ============================================================

client = chromadb.Client()


# ============================================================
# 6. REMOVE OLD COLLECTION IF IT EXISTS
# ============================================================

collection_name = "notes_20"

try:
    client.delete_collection(
        collection_name
    )

    print(
        f"\nDeleted old collection: {collection_name}"
    )

except Exception:
    print(
        "\nNo old collection found."
    )


# ============================================================
# 7. CREATE NEW COLLECTION
# ============================================================

col = client.create_collection(
    name=collection_name,
    metadata={
        "hnsw:space": "cosine"
    }
)

print(
    f"Created collection: {collection_name}"
)


# ============================================================
# 8. ADD CHUNKS + EMBEDDINGS TO CHROMA
# ============================================================

col.add(
    ids=[
        str(i)
        for i in range(len(chunks))
    ],

    embeddings=chunk_embeddings.tolist(),

    documents=chunks
)

print(
    "\nAdded",
    col.count(),
    "documents to Chroma."
)


# ============================================================
# 9. SEARCH
# ============================================================

q = "staff and fresh food"

print("\nSearching for:")
print(q)


# Turn query into a vector
query_embedding = embed(q)


# Search Chroma
results = col.query(
    query_embeddings=query_embedding.tolist(),
    n_results=2
)


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print("\n========== SEARCH RESULTS ==========")

for i, document in enumerate(
    results["documents"][0],
    start=1
):

    distance = results["distances"][0][i - 1]

    print(
        f"\nResult {i}"
    )

    print(
        "Distance:",
        round(distance, 4)
    )

    print(
        "Document:"
    )

    print(document)


print("\n====================================")

Number of chunks: 30

Chunk 0:
the food was good and the service was fast

Chunk 1:
good food and very friendly staff

Chunk 2:
great service and great value for money

Chunk 3:
the momo was delicious and the staff were friendly

Chunk 4:
delicious food, fast service, good price

Chunk 5:
excellent food and excellent service

Chunk 6:
the curry was delicious and the staff was kind

Chunk 7:
friendly staff and fresh food

Chunk 8:
fresh ingredients and good flavour

Chunk 9:
the dal bhat was delicious and hot

Chunk 10:
good value and the service was quick

Chunk 11:
quick service and delicious coffee

Chunk 12:
the staff was friendly and the food was fresh

Chunk 13:
great flavour and a clean place

Chunk 14:
clean tables and very good food

Chunk 15:
the thukpa was excellent and hot

Chunk 16:
excellent value, the food was fresh

Chunk 17:
we loved the food, service was fast

Chunk 18:
the biryani was delicious and the portion was generous

Chunk 19:
generous portion and good price

C

In [6]:
results

{'ids': [['20', '1']],
 'embeddings': None,
 'documents': [['the tea was good and the staff smiled',
   'good food and very friendly staff']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[5.161762237548828e-05, 0.0010559558868408203]]}

In [7]:


client = chromadb.Client()

col = client.get_collection("notes_20")


q = "staff and fresh food"

results = col.query(
    query_embeddings=query_embedding.tolist(),
    n_results=2
)


results


{'ids': [['20', '1']],
 'embeddings': None,
 'documents': [['the tea was good and the staff smiled',
   'good food and very friendly staff']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[5.161762237548828e-05, 0.0010559558868408203]]}

In [8]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# STEP 1: THE DATABASE (CHUNKING)
# ==========================================
# Imagine this is a textbook that we chopped into 5 paragraphs (chunks).
# The computer will automatically assign them IDs: 0, 1, 2, 3, 4.
chunks = [
    "Overfitting happens when a machine learning model memorizes the training data.", # ID 0
    "A database is used to store data safely.",                                       # ID 1
    "The learning rate controls how big of a step the model takes during training.",  # ID 2
    "Python is a popular programming language.",                                      # ID 3
    "A transformer is a deep learning architecture using attention mechanisms."       # ID 4
]

# ==========================================
# STEP 2: THE EMBEDDER (TRANSLATION)
# ==========================================
# We create a machine to turn English words into math (numbers).
embedder = TfidfVectorizer()

# We pass all our chunks through the machine to create our "Database Embeddings".
database_embeddings = embedder.fit_transform(chunks)


# ==========================================
# STEP 3: THE SEARCH ENGINE (THE KITCHEN)
# ==========================================
def ranked(q):
    """Takes a question, does similarity math, and returns a sorted list of Chunk IDs."""
    
    # 1. Turn the user's question into math
    q_vec = embedder.transform([q])
    
    # 2. Compare the question to every chunk in the database (Cosine Similarity)
    scores = cosine_similarity(q_vec, database_embeddings)[0]
    
    # 3. Sort the scores from highest match to lowest match
    sorted_ids = np.argsort(-scores) 
    
    # 4. Hand back the list of IDs (e.g., [2, 0, 4, 1, 3])
    return sorted_ids.tolist()


# ==========================================
# STEP 4: THE ANSWER KEY (THE HUMAN GRADER)
# ==========================================
# A human read the chunks above and wrote down the perfect answers.
EVAL = [
    ("what is overfitting?", 0),                     # The answer is in Chunk 0
    ("what does the learning rate control?", 2),     # The answer is in Chunk 2
    ("what is a transformer?", 4)                    # The answer is in Chunk 4
]


# ==========================================
# STEP 5: THE METRICS (RECALL & MRR)
# ==========================================
def rank_of_gold(q, gold):
    """Finds what position the Search Engine put the correct answer in."""
    retrieved_list = ranked(q)
    
    # If the search engine totally failed, give it an infinite rank
    if gold not in retrieved_list:
        return float('inf')
        
    # Find the index of the gold chunk, and add 1 (because humans count from 1)
    return retrieved_list.index(gold) + 1


# --- Let's run the actual test! ---

k = 3 # We only have patience to look at the Top 3 results

# Notice the square brackets [] to create a completed list before doing the math!
recall_list = [rank_of_gold(q, gold) <= k for q, gold in EVAL]
mrr_list    = [1 / rank_of_gold(q, gold) for q, gold in EVAL]

final_recall = np.mean(recall_list)
final_mrr    = np.mean(mrr_list)

print("--- AI SEARCH ENGINE REPORT CARD ---")
print(f"Final Recall@{k} Score: {final_recall * 100}%")
print(f"Final MRR Score:      {final_mrr}")

--- AI SEARCH ENGINE REPORT CARD ---
Final Recall@3 Score: 100.0%
Final MRR Score:      1.0


In [9]:
q = "what is overfitting?"

def ranked(q):
    """Takes a question, does similarity math, and returns a sorted list of Chunk IDs."""
    
    # 1. Turn the user's question into math
    q_vec = embedder.transform([q])
    
    # 2. Compare the question to every chunk in the database (Cosine Similarity)
    scores = cosine_similarity(q_vec, database_embeddings)[0]
    
    # 3. Sort the scores from highest match to lowest match
    sorted_ids = np.argsort(-scores) 
    
    # 4. Hand back the list of IDs (e.g., [2, 0, 4, 1, 3])
    return sorted_ids.tolist()


    

In [10]:
# ============================================================
# CHROMA METADATA FILTERING — FIXED VERSION
# ============================================================

import chromadb
from sklearn.feature_extraction.text import TfidfVectorizer


# ============================================================
# STEP 1: THE DATA
# ============================================================

chunks = [
    "Oak trees grow very tall and drop leaves.",
    "Pine trees keep their green needles all year.",
    "To get stronger, you must lift heavy weights.",
    "Cardio training improves your heart health.",
    "A transformer is an advanced AI architecture."
]


# ============================================================
# STEP 2: METADATA
# ============================================================
# Each document gets a label/topic.

metadatas = [
    {"topic": "trees"},
    {"topic": "trees"},
    {"topic": "training"},
    {"topic": "training"},
    {"topic": "ai"}
]


# ============================================================
# STEP 3: CREATE THE EMBEDDER
# ============================================================

vectorizer = TfidfVectorizer()

chunk_embeddings = vectorizer.fit_transform(
    chunks
).toarray()


# ============================================================
# STEP 4: QUERY EMBEDDING FUNCTION
# ============================================================

def embed(text):
    """
    Turn one piece of text into a vector.
    """

    return vectorizer.transform(
        [text]
    ).toarray()[0]


# ============================================================
# STEP 5: START CHROMA
# ============================================================

client = chromadb.Client()


# ============================================================
# STEP 6: GET OR CREATE COLLECTION
# ============================================================
# IMPORTANT:
#
# create_collection()
#     → ERROR if it already exists
#
# get_or_create_collection()
#     → uses existing collection
#     → creates it if it doesn't exist
#
# This makes the notebook safe to rerun.

col = client.get_or_create_collection(
    name="notes_2",
    metadata={
        "hnsw:space": "cosine"
    }
)


# ============================================================
# STEP 7: CLEAR OLD DATA
# ============================================================
# This makes rerunning the notebook safe.
#
# Without this, Chroma may complain that IDs such as "0"
# already exist.

old_data = col.get()

old_ids = old_data.get("ids", [])

if old_ids:

    col.delete(
        ids=old_ids
    )

    print(
        "Removed old documents:",
        len(old_ids)
    )


# ============================================================
# STEP 8: CREATE IDS
# ============================================================

ids = [
    str(i)
    for i in range(len(chunks))
]


# ============================================================
# STEP 9: ADD DATA TO CHROMA
# ============================================================

col.add(
    ids=ids,
    embeddings=chunk_embeddings.tolist(),
    documents=chunks,
    metadatas=metadatas
)

print(
    "Documents stored:",
    col.count()
)


# ============================================================
# STEP 10: ASK A QUESTION
# ============================================================

question = "How do I build muscle?"

print("\nQuestion:")
print(question)


# ============================================================
# STEP 11: SEARCH WITH METADATA FILTER
# ============================================================
# We ONLY want documents whose topic is "training".

query_embedding = embed(
    question
)

results = col.query(
    query_embeddings=[
        query_embedding.tolist()
    ],

    n_results=2,

    where={
        "topic": "training"
    }
)


# ============================================================
# STEP 12: SHOW RESULTS
# ============================================================

print("\n--- SEARCH RESULTS ---")

for i, doc in enumerate(
    results["documents"][0],
    start=1
):

    print(
        f"{i}. {doc}"
    )


# ============================================================
# STEP 13: SHOW METADATA TOO
# ============================================================

print("\n--- METADATA ---")

for metadata in results["metadatas"][0]:

    print(
        metadata
    )

Documents stored: 5

Question:
How do I build muscle?

--- SEARCH RESULTS ---
1. To get stronger, you must lift heavy weights.
2. Cardio training improves your heart health.

--- METADATA ---
{'topic': 'training'}
{'topic': 'training'}


In [11]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# STEP 1: THE DATABASE
# ==========================================
documents = [
    "The quick sports car",
    "A fast food delivery vehicle",
    "The fastest racing vehicle"
]

# ==========================================
# STEP 2: THE TWO DETECTIVES
# ==========================================
# Detective Sparse: Only cares about exact letter-for-letter keyword matches.
sparse_machine = CountVectorizer()
sparse_db = sparse_machine.fit_transform(documents)

# Detective Dense: Cares about the "weight" and meaning of the words (TF-IDF).
# (In a real system, you would use SVD or an LLM Embedding here).
dense_machine = TfidfVectorizer()
dense_db = dense_machine.fit_transform(documents)


# ==========================================
# STEP 3: THE NORMALIZER (The Peacemaker)
# ==========================================
def norm(scores):
    """
    Forces all scores to live perfectly between 0.0 and 1.0.
    If the highest score is 50, it divides everything by 50 so the winner is 1.0.
    """
    max_score = np.max(scores)
    if max_score == 0:
        return scores # Prevent dividing by zero if there are no matches!
    return scores / max_score


# ==========================================
# STEP 4: THE HYBRID BOSS (The Blender)
# ==========================================
def hybrid_search(query, alpha=0.5):
    print(f"\n--- Searching for: '{query}' (Alpha: {alpha}) ---")
    
    # 1. Get raw scores from Detective Dense
    q_dense = dense_machine.transform([query])
    raw_dense_scores = cosine_similarity(q_dense, dense_db)[0]
    
    # 2. Get raw scores from Detective Sparse
    q_sparse = sparse_machine.transform([query])
    raw_sparse_scores = cosine_similarity(q_sparse, sparse_db)[0]
    
    # 3. Normalize both so they are fair (0.0 to 1.0)
    dense_clean = norm(raw_dense_scores)
    sparse_clean = norm(raw_sparse_scores)
    
    # 4. THE MAGIC BLEND FORMULA
    # alpha controls Dense, (1 - alpha) controls Sparse
    final_scores = (alpha * dense_clean) + ((1 - alpha) * sparse_clean)
    
    # 5. Sort the results from highest to lowest
    winning_ids = np.argsort(-final_scores)
    
    # Print the leaderboard
    for rank, doc_id in enumerate(winning_ids):
        score = final_scores[doc_id]
        print(f"Rank {rank+1} | Score: {score:.2f} | Doc: {documents[doc_id]}")


# ==========================================
# STEP 5: RUNNING THE SIMULATION
# ==========================================
user_query = "fast vehicle"

# Scenario A: 50/50 Blend (A balanced approach)
hybrid_search(user_query, alpha=0.5)

# Scenario B: 100% Dense (Only listen to the Meaning Detective)
hybrid_search(user_query, alpha=1.0)

# Scenario C: 100% Sparse (Only listen to the Keyword Detective)
hybrid_search(user_query, alpha=0.0)


--- Searching for: 'fast vehicle' (Alpha: 0.5) ---
Rank 1 | Score: 1.00 | Doc: A fast food delivery vehicle
Rank 2 | Score: 0.45 | Doc: The fastest racing vehicle
Rank 3 | Score: 0.00 | Doc: The quick sports car

--- Searching for: 'fast vehicle' (Alpha: 1.0) ---
Rank 1 | Score: 1.00 | Doc: A fast food delivery vehicle
Rank 2 | Score: 0.39 | Doc: The fastest racing vehicle
Rank 3 | Score: 0.00 | Doc: The quick sports car

--- Searching for: 'fast vehicle' (Alpha: 0.0) ---
Rank 1 | Score: 1.00 | Doc: A fast food delivery vehicle
Rank 2 | Score: 0.50 | Doc: The fastest racing vehicle
Rank 3 | Score: 0.00 | Doc: The quick sports car


In [12]:
import pypdf
from openai import OpenAI
import sys
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import fitz
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

pdf_path = (r"E:\AI-ML\day 32\AI Engineering & Machine Learning - Full Syllabus.pdf")

doc = fitz.open(pdf_path)
text = ""
for page in doc :
    text += page.get_text() + "\n"


chunk_size = 1000
overlap = 30

chunks = []

start = 0
while start<len(text):
    end = start + chunk_size
    chunks.append(text[start:end])
    start += chunk_size - overlap

# for i, chunk in enumerate(chunks):
#     print(f"\n------Chunk {i}------")
#     print(chunk)


In [13]:
chunks

['SAARATHI ACADEMY\nfor Digital Excellence\nAI Engineering & Machine Learning\nFULL SYLLABUS\nSKILLS & TOOLS YOU WILL MASTER\nPython + scikit-learn\nPyTorch + Transformers\nLangChain + LangGraph\nPC RAG + Pinecone\nAM Agents + Mem0 + MCP\nFastAPI + Docker\n10 Reasons This Works.\nWhy students choose Saarathi - and how each card helps you ship a real career.\nsaarathiacademy.com.np/ai-engineering-and-machine-learning-\ncourse-in-nepal\n+977-9744442469 \xa0·\xa0 +977-\n9761095364\nTR-10\n01\n01\n02\n02\n03\n03\n04\n04\n05\n05\n06\n06\n07\n07\n08\n08\n09\n09\n10\n10\n10 Max / Batch\nNot 50. Not 30. Ten. Every\nstudent seen, heard, and\nunblocked - every class.\nSunday\nOpen\nClassroom\nEvery Sunday the\nclassroom \nis\nopen. Come in,\npair-program,\nget unstuck, or\njust focus.\nWeekly\nDetailed\nSyllabus\nFull syllabus from\nday one. Every\nweek: \nwhat \nto\nread, \nwhat \nto\nbuild, what the\noutcome is.\nRevision Every 5th\nDay\nEvery fifth class is a revision\nsession. Nothing piles 

In [15]:
import os
import json
import chromadb

from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer


# ============================================================
# 1. DOCUMENTS
# ============================================================

chunks = [
    "RAG combines retrieval with a language model to answer questions using external documents.",

    "Machine learning models learn patterns from training data and use those patterns to make predictions.",

    "A neural network contains layers of neurons. Each layer transforms information before passing it to the next layer.",

    "Overfitting happens when a model memorizes training data too closely and performs poorly on unseen data.",

    "Embeddings represent words, sentences, or documents as vectors of numbers.",

    "Chroma is a vector database that can store embeddings and retrieve similar documents.",

    "TF-IDF converts text into numerical vectors based on how important words are in a document.",

    "Cosine similarity measures how similar two vectors point in the same direction.",

    "Python is commonly used for machine learning because it has libraries such as NumPy, pandas, and scikit-learn.",

    "An AI/ML engineer builds, trains, evaluates, deploys, and maintains machine learning systems."
]


# ============================================================
# 2. GET GEMINI API KEY
# ============================================================

api_key = os.getenv("AQ.Ab8RN6J7VvI96jnVRqNMXeFfszfElHvhueCVACBcOt7BmHiMWg")

if api_key is None or api_key.strip() == "":
    raise RuntimeError(
        "GEMINI_API_KEY is not set. "
        "Set it in PowerShell, then restart Jupyter."
    )

print("Gemini API key found.")
print("Key is NOT displayed.")

# ============================================================
# 3. CONNECT TO GEMINI
# ============================================================

llm = OpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

print("Gemini client connected.")


# ============================================================
# 4. TOPICS
# ============================================================

TOPICS = [
    "RAG",
    "AI_ML",
    "Course",
    "Academy"
]


# ============================================================
# 5. GEMINI CREATES METADATA
# ============================================================

def generate_metadata(chunks):

    numbered = "\n\n".join(
        f"[{i}] {chunk}"
        for i, chunk in enumerate(chunks)
    )

    prompt = f"""
You are organizing documents for a RAG system.

For every document:

- Select exactly one topic from:
{TOPICS}

- Write a short one-sentence summary.

Return JSON ONLY.

Required format:

{{
    "items": [
        {{
            "topic": "AI_ML",
            "summary": "Short summary."
        }}
    ]
}}

There must be exactly one item for every document.

DOCUMENTS:

{numbered}
"""

    response = llm.chat.completions.create(
        model="gemini-3.5-flash-lite",
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    text = response.choices[0].message.content.strip()

    # Remove Markdown code fences if returned
    if text.startswith("```json"):
        text = text[7:]

    elif text.startswith("```"):
        text = text[3:]

    if text.endswith("```"):
        text = text[:-3]

    text = text.strip()

    data = json.loads(text)

    items = data["items"]

    if len(items) != len(chunks):
        raise ValueError(
            f"Expected {len(chunks)} metadata items, "
            f"but Gemini returned {len(items)}."
        )

    metadatas = []

    for item in items:

        topic = item.get("topic", "Course")

        if topic not in TOPICS:
            topic = "Course"

        summary = str(
            item.get("summary", "")
        )[:300]

        metadatas.append({
            "topic": topic,
            "summary": summary
        })

    return metadatas


# ============================================================
# 6. GENERATE METADATA
# ============================================================

print("\nGenerating metadata with Gemini...")

metadatas = generate_metadata(chunks)


# ============================================================
# 7. DISPLAY METADATA
# ============================================================

print("\n========== GEMINI METADATA ==========")

for i, (chunk, metadata) in enumerate(
    zip(chunks, metadatas)
):

    print(f"\nDocument {i}")
    print("Topic   :", metadata["topic"])
    print("Summary :", metadata["summary"])
    print("Text    :", chunk)


# ============================================================
# 8. TF-IDF EMBEDDINGS
# ============================================================

vectorizer = TfidfVectorizer(
    stop_words="english"
)

chunk_embeddings = vectorizer.fit_transform(
    chunks
).toarray()

print(
    "\nEmbedding shape:",
    chunk_embeddings.shape
)


# ============================================================
# 9. EMBED FUNCTION
# ============================================================

def embed(text):

    return vectorizer.transform(
        [text]
    ).toarray()[0]


# ============================================================
# 10. CHROMA
# ============================================================

client = chromadb.Client()

collection_name = "day32_notes"


# IMPORTANT:
# get_or_create_collection prevents
# "Collection already exists"

col = client.get_or_create_collection(
    name=collection_name,
    metadata={
        "hnsw:space": "cosine"
    }
)


# ============================================================
# 11. REMOVE OLD DATA
# ============================================================

existing = col.get()

existing_ids = existing.get("ids", [])

if existing_ids:

    col.delete(
        ids=existing_ids
    )

    print(
        f"\nRemoved {len(existing_ids)} old documents."
    )


# ============================================================
# 12. STORE DOCUMENTS
# ============================================================

ids = [
    str(i)
    for i in range(len(chunks))
]

col.add(
    ids=ids,
    embeddings=chunk_embeddings.tolist(),
    documents=chunks,
    metadatas=metadatas
)

print(
    "\nDocuments stored:",
    col.count()
)


# ============================================================
# 13. SEARCH
# ============================================================

question = "How do I become an AI ML engineer?"

print("\nQUESTION:")
print(question)

query_embedding = embed(question)


results = col.query(
    query_embeddings=[
        query_embedding.tolist()
    ],
    n_results=3
)


# ============================================================
# 14. SEARCH RESULTS
# ============================================================

print("\n========== SEARCH RESULTS ==========")

for i, document in enumerate(
    results["documents"][0],
    start=1
):

    print(f"\nResult {i}")
    print("Document:", document)

    print(
        "Metadata:",
        results["metadatas"][0][i - 1]
    )

    print(
        "Distance:",
        round(
            results["distances"][0][i - 1],
            4
        )
    )


# ============================================================
# 15. FILTERED SEARCH
# ============================================================

print("\n========== AI/ML FILTERED SEARCH ==========")

filtered_results = col.query(
    query_embeddings=[
        query_embedding.tolist()
    ],
    n_results=3,
    where={
        "topic": "AI_ML"
    }
)


for i, document in enumerate(
    filtered_results["documents"][0],
    start=1
):

    print(
        f"\nResult {i}:"
    )

    print(document)

    print(
        "Metadata:",
        filtered_results["metadatas"][0][i - 1]
    )


print("\n========== DAY 32 COMPLETE ==========")

RuntimeError: GEMINI_API_KEY is not set. Set it in PowerShell, then restart Jupyter.

In [ ]:
results

In [ ]:
results['documents']

In [ ]:


def make_answer(user_query: str):


    results = col.query(
    query_embeddings=[embed(user_query).tolist()],
    n_results=2,
    where={"topic": "Academy"}
    ) 

    chunks = results['documents']
    
    SYSTEM_PROMPT = f"""

    You will be given context in chunks and user query now answer the question by reading context with factual data and relevancy. Give proper explanation along different points in bullet forms

    DO NOT USE \n for line breaking. It should be proper markdown format
    context: {chunks}
    
    
    """

    resp = llm.chat.completions.create(
        model="gemini-3.6-flash",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": SYSTEM_PROMPT}, 
                  {"role":"user", "content":user_query}
                 ]
        
    )

    resp = resp.choices[0].message.content

    return resp
    

In [ ]:
answer = make_answer("How do i become a aiml engineer?")

In [ ]:
answer

In [ ]:
neat_res = json.loads(answer)

In [ ]:
neat_res

Based on the provided context, here is how you can become an AI/ML Engineer by following a structured learning path:',
 'steps': ['**Master Core Skills & Tools**: Focus on gaining expertise in essential technologies including Python, scikit-learn, PyTorch, Transformers, LangChain, LangGraph, Pinecone (for RAG), Agents (Mem0, MCP), FastAPI, and Docker.',
  '**Build Deployed Portfolio Projects**: Instead of creating basic toy projects, build and deploy public, production-ready artefacts for every module to showcase your practical capabilities.',
  '**Follow a Structured Syllabus**: Stick to a clear day-by-day curriculum that clearly defines what to read, what to build, and the expected outcomes, incorporating regular revision sessions (e.g., every 5th day) to ensure continuous retention.',
  '**Participate in Mentorship & Pair-Programming**: Work in small cohorts to receive direct mentor feedback, project reviews, and engage in open classroom sessions to pair-program and solve complex challenges.',
  '**Career Preparation**: Refine your professional profile by polishing your CV and LinkedIn, participating in mock interviews, and leveraging referral networks.']

In [ ]:
"""

Make a class called retriver

need 3 methods

1. chunker: takes the document and chunks it
2. embedder: takes the chunks and embed it
3. search: takes the query and do similarity search



"""


# ============================================================
# DAY 32 PRACTICE — RETRIEVER CLASS
# ============================================================
# The goal:
#   1. chunker(document) -> makes small pieces
#   2. embedder(chunks) -> turns text into vectors
#   3. search(query) -> finds the most similar chunks
#
# This version is deliberately simple so you can understand it.
# It uses TF-IDF + cosine similarity.

import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


class retriver:

    def __init__(self, chunk_size=80, overlap=15):
        self.chunk_size = chunk_size
        self.overlap = overlap

        # These will be created by embedder().
        self.vectorizer = None
        self.embeddings = None
        self.chunks = None

    # --------------------------------------------------------
    # 1. CHUNKER
    # --------------------------------------------------------
    def chunker(self, document):
        """
        Take one large document and split it into overlapping chunks.
        """

        if not isinstance(document, str):
            raise TypeError("document must be a string")

        words = document.split()

        if not words:
            return []

        if self.chunk_size <= self.overlap:
            raise ValueError("chunk_size must be greater than overlap")

        step = self.chunk_size - self.overlap

        chunks = []

        for start in range(0, len(words), step):
            piece = words[start:start + self.chunk_size]

            if piece:
                chunks.append(" ".join(piece))

            if start + self.chunk_size >= len(words):
                break

        return chunks

    # --------------------------------------------------------
    # 2. EMBEDDER
    # --------------------------------------------------------
    def embedder(self, chunks):
        """
        Turn chunks into TF-IDF vectors.
        """

        if not chunks:
            raise ValueError("chunks cannot be empty")

        self.chunks = list(chunks)

        self.vectorizer = TfidfVectorizer(
            stop_words="english"
        )

        self.embeddings = self.vectorizer.fit_transform(
            self.chunks
        )

        return self.embeddings

    # --------------------------------------------------------
    # 3. SEARCH
    # --------------------------------------------------------
    def search(self, query, k=3):
        """
        Turn the query into a vector, compare it with every chunk,
        and return the best matches.
        """

        if self.vectorizer is None:
            raise RuntimeError(
                "Run embedder(chunks) before search()."
            )

        if not isinstance(query, str) or not query.strip():
            raise ValueError("query must be a non-empty string")

        k = min(k, len(self.chunks))

        query_vector = self.vectorizer.transform(
            [query]
        )

        scores = cosine_similarity(
            query_vector,
            self.embeddings
        )[0]

        ranked_indices = np.argsort(
            -scores
        )[:k]

        results = []

        for rank, index in enumerate(
            ranked_indices,
            start=1
        ):
            results.append({
                "rank": rank,
                "score": float(scores[index]),
                "text": self.chunks[index]
            })

        return results


# ============================================================
# DEMO
# ============================================================

document = """
Machine learning allows computers to learn patterns from data.
Training data is used to teach a model. Test data is used to
check how well the model works on unseen examples. Overfitting
happens when a model memorizes training examples too closely.
Regularization can help reduce overfitting. A random forest
combines many decision trees to make predictions. Embeddings
represent text as vectors of numbers and can be used for search.
RAG retrieves useful context before asking a language model
to generate an answer.
"""

retriever = retriver(
    chunk_size=25,
    overlap=5
)

chunks = retriever.chunker(document)

print("NUMBER OF CHUNKS:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\nCHUNK {i}:")
    print(chunk)

retriever.embedder(chunks)

query = "How can we reduce overfitting?"

results = retriever.search(
    query,
    k=3
)

print("\nSEARCH RESULTS")

for result in results:
    print(
        f"\nRank {result['rank']} | "
        f"Score {result['score']:.3f}"
    )
    print(result["text"])


In [ ]:
"""

Make a neural network (class network) custom methods for training and prediction.

4 layers use sequnetial to encapulate all the layers in one attribute


task: handwritten digits 

MNIST handwritten dataset. 


"""


# ============================================================
# DAY 32 PRACTICE — 4-LAYER NEURAL NETWORK
# ============================================================
# Task: handwritten digit classification.
#
# We use the MNIST dataset:
#   28 x 28 grayscale image = 784 numbers
#   output = one of 10 digits (0...9)
#
# Four Linear layers:
#   784 -> 128
#   128 -> 64
#   64  -> 32
#   32  -> 10
#
# ReLU is used between the layers.
#
# "Sequential" puts the layers into one object.
# Custom methods:
#   fit()     -> training
#   predict() -> prediction
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


class network(nn.Module):

    def __init__(self):
        super().__init__()

        # One attribute contains the complete 4-layer network.
        self.layers = nn.Sequential(

            # Layer 1
            nn.Linear(28 * 28, 128),
            nn.ReLU(),

            # Layer 2
            nn.Linear(128, 64),
            nn.ReLU(),

            # Layer 3
            nn.Linear(64, 32),
            nn.ReLU(),

            # Layer 4
            nn.Linear(32, 10)

        )

    def forward(self, x):
        """
        PyTorch calls forward() when we pass data through
        the network.
        """

        # Flatten:
        # [batch, 1, 28, 28]
        # becomes
        # [batch, 784]
        x = x.view(
            x.size(0),
            -1
        )

        return self.layers(x)

    def fit(
        self,
        train_loader,
        epochs=3,
        learning_rate=0.001,
        device=None
    ):
        """
        Train the neural network.
        """

        if device is None:
            device = (
                "cuda"
                if torch.cuda.is_available()
                else "cpu"
            )

        self.to(device)

        loss_function = nn.CrossEntropyLoss()

        optimizer = torch.optim.Adam(
            self.parameters(),
            lr=learning_rate
        )

        for epoch in range(epochs):

            self.train()

            total_loss = 0.0
            correct = 0
            total = 0

            for images, labels in train_loader:

                images = images.to(device)
                labels = labels.to(device)

                # Remove old gradients.
                optimizer.zero_grad()

                # Make predictions.
                outputs = self(images)

                # Calculate how wrong we are.
                loss = loss_function(
                    outputs,
                    labels
                )

                # Work out how to change weights.
                loss.backward()

                # Update weights.
                optimizer.step()

                total_loss += (
                    loss.item()
                    *
                    images.size(0)
                )

                predictions = (
                    outputs.argmax(dim=1)
                )

                correct += (
                    predictions == labels
                ).sum().item()

                total += labels.size(0)

            average_loss = (
                total_loss / total
            )

            accuracy = (
                correct / total
            )

            print(
                f"Epoch {epoch + 1}/{epochs} | "
                f"loss={average_loss:.4f} | "
                f"accuracy={accuracy:.4f}"
            )

        return self

    @torch.no_grad()
    def predict(
        self,
        images,
        device=None
    ):
        """
        Predict digit labels for images.
        """

        if device is None:
            device = (
                "cuda"
                if torch.cuda.is_available()
                else "cpu"
            )

        self.eval()
        self.to(device)

        images = images.to(device)

        outputs = self(images)

        predictions = outputs.argmax(
            dim=1
        )

        return predictions.cpu()


# ============================================================
# LOAD MNIST
# ============================================================

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)


# ============================================================
# CREATE + TRAIN
# ============================================================

model = network()

model.fit(
    train_loader,
    epochs=3,
    learning_rate=0.001
)


# ============================================================
# TEST ACCURACY
# ============================================================

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model.to(device)
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        predictions = model.predict(
            images,
            device=device
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

test_accuracy = correct / total

print(
    f"\nTest accuracy: {test_accuracy:.4f}"
)


# ============================================================
# SHOW A FEW PREDICTIONS
# ============================================================

images, labels = next(
    iter(test_loader)
)

predictions = model.predict(
    images[:10],
    device=device
)

print("\nFIRST 10 PREDICTIONS")

for i in range(10):

    print(
        f"Image {i}: "
        f"true={labels[i].item()} | "
        f"predicted={predictions[i].item()}"
    )
